# Mixture of Experts in LLMs: From Switch to DeepSeek-V3

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/llm/sparse_mixture_of_experts.ipynb)

Companion notebook to the [blog post](https://sesen.ai/blog/mixture-of-experts-llms-sparse-routing). Tiny sparse MoE in PyTorch, all five experiments:

1. Sparse MoE with top-2 gating on 4-blob data — experts crystallise on regions
2. Switch Transformer load-balancing loss vs collapse (without aux loss)
3. Dense MLP vs sparse MoE at matched active compute
4. Top-1 (Switch) vs Top-2 (Mixtral) routing trade-off
5. DeepSeek-V3-style auxiliary-loss-free balancing

All experiments run on CPU in seconds — no GPU, no HuggingFace, no model downloads.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
from sklearn.model_selection import train_test_split

torch.manual_seed(0)
np.random.seed(0)

## 1. Sparse MoE in 30 lines of PyTorch

An expert is just an MLP. The MoE is a list of experts plus a small linear gating network. Top-k routing picks which experts run for each input.

In [ ]:
class Expert(nn.Module):
    def __init__(self, d_in, d_hidden, d_out):
        super().__init__()
        self.fc1 = nn.Linear(d_in, d_hidden)
        self.fc2 = nn.Linear(d_hidden, d_out)
    def forward(self, x):
        return self.fc2(F.relu(self.fc1(x)))

class SparseMoE(nn.Module):
    def __init__(self, d_in, d_hidden, d_out, n_experts=4, top_k=2,
                 aux_loss_weight=0.01):
        super().__init__()
        self.n_experts, self.top_k = n_experts, top_k
        self.aux_loss_weight = aux_loss_weight
        self.gate = nn.Linear(d_in, n_experts)
        self.experts = nn.ModuleList([Expert(d_in, d_hidden, d_out)
                                      for _ in range(n_experts)])
    def forward(self, x):
        gate_probs = F.softmax(self.gate(x), dim=-1)
        topk_vals, topk_idx = torch.topk(gate_probs, self.top_k, dim=-1)
        topk_w = topk_vals / (topk_vals.sum(-1, keepdim=True) + 1e-9)
        out = torch.zeros(x.size(0), self.experts[0].fc2.out_features)
        for k in range(self.top_k):
            for e in range(self.n_experts):
                m = (topk_idx[:, k] == e)
                if m.any():
                    out[m] = out[m] + topk_w[m, k:k+1] * self.experts[e](x[m])
        # Switch aux loss (Fedus et al. 2022)
        f_i = torch.bincount(topk_idx[:, 0], minlength=self.n_experts).float()
        f_i = f_i / f_i.sum()
        P_i = gate_probs.mean(0)
        aux_loss = self.n_experts * (f_i * P_i).sum()
        return out, aux_loss

In [ ]:
# 4-region 2D blob data — 4 well-separated clusters, one class each
centers = [(-2.5, -2.5), (2.5, -2.5), (-2.5, 2.5), (2.5, 2.5)]
X, y = make_blobs(n_samples=2000, centers=centers, cluster_std=0.8, random_state=0)
X = X.astype(np.float32); y = y.astype(np.int64)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=0)

torch.manual_seed(0)
moe = SparseMoE(d_in=2, d_hidden=8, d_out=4, n_experts=4, top_k=2,
                aux_loss_weight=0.01)
opt = torch.optim.Adam(moe.parameters(), lr=0.01)
Xt = torch.from_numpy(X_tr); yt = torch.from_numpy(y_tr)
for epoch in range(80):
    opt.zero_grad()
    logits, aux = moe(Xt)
    loss = F.cross_entropy(logits, yt) + moe.aux_loss_weight * aux
    loss.backward(); opt.step()

with torch.no_grad():
    test_logits, _ = moe(torch.from_numpy(X_te))
    print(f'Test accuracy: {float((test_logits.argmax(-1).numpy() == y_te).mean()):.4f}')

In [ ]:
# Visualise the top-1 expert territory each gate has carved out
from matplotlib.colors import ListedColormap
EXPERT_COLORS = ['#2d6cdf', '#cc3344', '#d4a24c', '#119955']
cmap = ListedColormap(EXPERT_COLORS)

x_min, x_max = X[:, 0].min()-0.5, X[:, 0].max()+0.5
y_min, y_max = X[:, 1].min()-0.5, X[:, 1].max()+0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))
grid = np.c_[xx.ravel(), yy.ravel()].astype(np.float32)
with torch.no_grad():
    top1 = F.softmax(moe.gate(torch.from_numpy(grid)), dim=-1).argmax(-1).numpy().reshape(xx.shape)
    top1_data = F.softmax(moe.gate(torch.from_numpy(X)), dim=-1).argmax(-1).numpy()

fig, ax = plt.subplots(figsize=(7, 5.5))
ax.contourf(xx, yy, top1, levels=np.arange(-0.5, 4.5), cmap=cmap, alpha=0.4)
ax.scatter(X[:, 0], X[:, 1], c=top1_data, cmap=cmap, s=14,
           edgecolors='white', linewidth=0.3)
ax.set_title('Top-1 expert territory after 80 epochs')
ax.set_aspect('equal', 'box')
plt.show()

## 2. Load balancing: with vs without aux loss

Without the Switch Transformer auxiliary loss, the gating network can collapse onto a few experts and leave the rest dead.

In [ ]:
def train_moe(use_aux, aux_weight=0.1):
    torch.manual_seed(2)
    m = SparseMoE(d_in=2, d_hidden=8, d_out=4, n_experts=4, top_k=1,
                  aux_loss_weight=aux_weight)
    opt = torch.optim.Adam(m.parameters(), lr=0.01)
    Xt = torch.from_numpy(X); yt = torch.from_numpy(y)
    for _ in range(80):
        opt.zero_grad()
        logits, aux = m(Xt)
        loss = F.cross_entropy(logits, yt)
        if use_aux:
            loss = loss + m.aux_loss_weight * aux
        loss.backward(); opt.step()
    with torch.no_grad():
        gate_probs = F.softmax(m.gate(Xt), dim=-1)
        top1 = gate_probs.argmax(-1).numpy()
        util = np.bincount(top1, minlength=4) / len(top1)
        acc = float((m(Xt)[0].argmax(-1).numpy() == y).mean())
    return util, acc

util_no, acc_no = train_moe(use_aux=False)
util_yes, acc_yes = train_moe(use_aux=True, aux_weight=0.1)
print(f'WITHOUT aux loss: utilisation = {util_no.round(3)},  acc = {acc_no:.4f}')
print(f'WITH aux loss:    utilisation = {util_yes.round(3)},  acc = {acc_yes:.4f}')

## 3. Dense MLP vs Sparse MoE at matched active compute

Same active parameters per forward pass; MoE has 2x total capacity in reserve.

In [ ]:
class DenseMLP(nn.Module):
    def __init__(self, d_in, d_hidden, d_out):
        super().__init__()
        self.fc1 = nn.Linear(d_in, d_hidden)
        self.fc2 = nn.Linear(d_hidden, d_out)
    def forward(self, x):
        return self.fc2(F.relu(self.fc1(x)))

# 8D, 4-class problem
rng = np.random.default_rng(3)
n, d_in = 6000, 8
centers8 = np.array([[3,3,0,0,0,0,0,0],[-3,-3,0,0,0,0,0,0],
                      [3,-3,0,0,0,0,0,0],[-3,3,0,0,0,0,0,0]], dtype=np.float32)
y8 = rng.integers(0, 4, n)
X8 = centers8[y8] + rng.normal(0, 1.2, (n, d_in)).astype(np.float32)
Q, _ = np.linalg.qr(rng.standard_normal((d_in, d_in)))
X8 = X8 @ Q.astype(np.float32).T
X8_tr, X8_te, y8_tr, y8_te = train_test_split(X8, y8, test_size=0.25, random_state=0)
y8_tr = y8_tr.astype(np.int64); y8_te = y8_te.astype(np.int64)

torch.manual_seed(0)
dense = DenseMLP(d_in, 32, 4)
opt = torch.optim.Adam(dense.parameters(), lr=0.01)
Xt = torch.from_numpy(X8_tr); yt = torch.from_numpy(y8_tr)
for _ in range(80):
    opt.zero_grad(); F.cross_entropy(dense(Xt), yt).backward(); opt.step()
with torch.no_grad():
    dense_acc = float((dense(torch.from_numpy(X8_te)).argmax(-1).numpy() == y8_te).mean())

torch.manual_seed(0)
moe = SparseMoE(d_in=d_in, d_hidden=16, d_out=4, n_experts=4, top_k=2,
                 aux_loss_weight=0.01)
opt = torch.optim.Adam(moe.parameters(), lr=0.01)
for _ in range(80):
    opt.zero_grad()
    logits, aux = moe(Xt)
    (F.cross_entropy(logits, yt) + moe.aux_loss_weight * aux).backward()
    opt.step()
with torch.no_grad():
    moe_acc = float((moe(torch.from_numpy(X8_te))[0].argmax(-1).numpy() == y8_te).mean())

dense_params = sum(p.numel() for p in dense.parameters())
moe_total = sum(p.numel() for p in moe.parameters())
moe_active = sum(p.numel() for p in moe.gate.parameters()) + 2 * sum(
    p.numel() for p in moe.experts[0].parameters())
print(f'Dense MLP:  acc = {dense_acc:.4f}, params = {dense_params}')
print(f'Sparse MoE: acc = {moe_acc:.4f}, total = {moe_total}, active/token = {moe_active}')

## 4. DeepSeek-V3 auxiliary-loss-free balancing

Maintain a per-expert routing bias updated by a moving-average rule based on observed utilisation. The optimiser doesn't see this bias, so it doesn't perturb the main gradient signal.

In [ ]:
class SparseMoEAuxFree(nn.Module):
    def __init__(self, d_in, d_hidden, d_out, n_experts=4, top_k=2, lr_bias=0.01):
        super().__init__()
        self.n_experts, self.top_k = n_experts, top_k
        self.lr_bias = lr_bias
        self.gate = nn.Linear(d_in, n_experts)
        self.experts = nn.ModuleList([Expert(d_in, d_hidden, d_out)
                                      for _ in range(n_experts)])
        # Bias added to gate logits; updated outside the optimiser
        self.register_buffer('bias', torch.zeros(n_experts))
    def forward(self, x):
        gate_logits = self.gate(x) + self.bias  # bias shifts routing
        gate_probs = F.softmax(gate_logits, dim=-1)
        topk_vals, topk_idx = torch.topk(gate_probs, self.top_k, dim=-1)
        topk_w = topk_vals / (topk_vals.sum(-1, keepdim=True) + 1e-9)
        out = torch.zeros(x.size(0), self.experts[0].fc2.out_features)
        for k in range(self.top_k):
            for e in range(self.n_experts):
                m = (topk_idx[:, k] == e)
                if m.any():
                    out[m] = out[m] + topk_w[m, k:k+1] * self.experts[e](x[m])
        return out, topk_idx
    def update_bias(self, topk_idx):
        with torch.no_grad():
            load = torch.bincount(topk_idx[:, 0], minlength=self.n_experts).float()
            load = load / load.sum()
            target = 1.0 / self.n_experts
            self.bias = self.bias + self.lr_bias * (target - load)

torch.manual_seed(2)
moe_af = SparseMoEAuxFree(d_in=2, d_hidden=8, d_out=4, n_experts=4, top_k=1)
opt = torch.optim.Adam(moe_af.parameters(), lr=0.01)
Xt = torch.from_numpy(X); yt = torch.from_numpy(y)
for _ in range(80):
    opt.zero_grad()
    logits, topk_idx = moe_af(Xt)
    F.cross_entropy(logits, yt).backward()
    opt.step()
    moe_af.update_bias(topk_idx)
with torch.no_grad():
    _, topk_idx = moe_af(Xt)
    util = np.bincount(topk_idx[:, 0].numpy(), minlength=4) / len(Xt)
    acc = float((moe_af(Xt)[0].argmax(-1).numpy() == y).mean())
print(f'Aux-loss-free MoE: utilisation = {util.round(3)}, acc = {acc:.4f}')
print(f'Final bias values: {moe_af.bias.numpy().round(3)}')

## Exercises

1. **Add shared experts**: Modify `SparseMoE` so that one of the `n_experts` is *always* active in addition to the top-k routed experts. Compare accuracy on the 8-D problem in section 3.

2. **Capacity factor**: Switch Transformer enforces a maximum number of tokens per expert per batch (the *capacity factor*). Add this constraint: if more tokens want to route to expert $e$ than its capacity allows, drop the excess (set their output to 0). Measure accuracy at capacity factors of 1.0, 1.25, 2.0.

3. **Compare aux-loss vs aux-loss-free at scale**: Generate 100-D data with 8 latent regimes and fit both `SparseMoE` (Switch aux loss) and `SparseMoEAuxFree`. Which gives higher held-out accuracy? Which gives more uniform expert utilisation?

4. **Token-level MoE in a Transformer**: Replace the FFN block of a small Transformer (use `nn.TransformerEncoderLayer` for the rest) with `SparseMoE`. Train on a toy seq2seq task. This is the architecture of Mixtral / DeepSeek at the per-block level.

5. **Routing entropy as a diagnostic**: Compute the entropy $H = -\sum_i p_i \log p_i$ of the gating distribution averaged over all tokens. Plot it over training epochs. What happens to entropy if you forget the aux loss?